# 🤖 Day 3: Transformers in Action, Sentiment Analysis with DistilBERT

**Goal:** Use a real **Transformer** model to read a sentence and decide if it is **positive 😊 or negative 😞**, and then open the model up to see *how* it works.

### What you will learn
1. What a Transformer is and why it changed AI
2. How to use a pre-trained model in **3 lines of code**
3. What happens inside: **tokenization → embeddings → attention → prediction**
4. How to *see* attention and find which words drive a decision
5. Where Transformers fail (sarcasm, bias)

### The one model we will use
**`distilbert-base-uncased-finetuned-sst-2-english`**, a small, fast version of Google's BERT, already trained to classify movie-review sentiment.

### How to use this notebook
- Run cells **one by one** with `Shift + Enter`, and read the explanation first.
- Look for **🧪 Try it yourself** boxes and change the text!
- No GPU needed. A normal Colab CPU is fine.

---
## 💡 Part 1: Theory in 5 minutes

### The old problem
Earlier models (like RNNs/LSTMs) read text **word by word, left to right**, like reading through a keyhole. By the time they reached the end of a long sentence, they had half-forgotten the beginning.

### The Transformer idea (2017, paper: *"Attention Is All You Need"*)
Instead of reading one word at a time, a Transformer looks at **all words at once** and uses **self-attention** to decide *which words matter to which other words*.

> **Example:** *"The animal didn't cross the street because **it** was too tired."*
> What does **"it"** refer to? Attention lets the model link **"it" → "animal"**.

### The pipeline we will build (step by step)

```
 "I loved this movie!"
          │
          ▼
 ┌──────────────────┐
 │  1. TOKENIZER     │  text → pieces (tokens) → ID numbers
 └──────────────────┘
          ▼
 ┌──────────────────┐
 │  2. EMBEDDINGS    │  each ID → a list of 768 numbers (its "meaning")
 └──────────────────┘
          ▼
 ┌──────────────────┐
 │  3. TRANSFORMER   │  6 layers of SELF-ATTENTION: words talk to each other
 │     LAYERS        │  and update their meaning based on context
 └──────────────────┘
          ▼
 ┌──────────────────┐
 │  4. CLASSIFIER    │  2 scores → softmax → NEGATIVE / POSITIVE
 └──────────────────┘
```

### The Transformer family

| Model | Type | Good at | Example |
|---|---|---|---|
| **BERT / DistilBERT** | Encoder | *Understanding* text (classification, search) | **← today** |
| **GPT** | Decoder | *Generating* text | ChatGPT |
| **T5 / BART** | Encoder + Decoder | Translation, summarization | Google Translate-style tasks |

**Pre-trained** means someone already trained it on huge amounts of text, so we don't start from zero. This is called **transfer learning**.

---
## 📦 Part 2: Setup

We use the **🤗 Hugging Face `transformers`** library. It gives easy access to thousands of pre-trained models. The first cell installs it (Colab often has it already, so it may say "already satisfied").

In [ ]:
!pip install -q transformers

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

print("Libraries loaded!")

---
## 🚀 Part 3: Magic in 3 lines, the `pipeline`

Hugging Face's **`pipeline`** hides all the complexity: it downloads the model, tokenizes your text, runs the model, and gives you a readable answer.

Run the next cell. The first time, it downloads the model (~250 MB), so give it a minute.

In [ ]:
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

classifier = pipeline("sentiment-analysis", model=MODEL_NAME)

result = classifier("I absolutely loved this workshop!")
print(result)

### 📖 Reading the output

```
[{'label': 'POSITIVE', 'score': 0.9998}]
```
- **label**: the model's decision
- **score**: its **confidence** (0 to 1). 0.9998 means about 99.98% sure.

Let's try several sentences at once:

In [ ]:
sentences = [
    "I absolutely loved this workshop!",
    "This was a complete waste of time.",
    "The food was okay, nothing special.",
    "What a fantastic, mind-blowing experience!",
    "The service was terrible and the staff were rude.",
]

results = classifier(sentences)

for text, res in zip(sentences, results):
    emoji = "😊" if res["label"] == "POSITIVE" else "😞"
    print(f"{emoji} {res['label']:8} ({res['score']*100:5.1f}%)  →  {text}")

### 🧪 Try it yourself
Type **your own sentence** below and see what the model thinks.

In [ ]:
my_text = "Learning AI is so much fun!"   # ✏️ change this sentence

print(classifier(my_text))

---
## 🔬 Part 4: Opening the black box

The `pipeline` is convenient, but a good AI engineer needs to know what's inside. We now load the **tokenizer** and the **model** separately.

We pass `attn_implementation="eager"` so the model is able to return its **attention weights** later.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, attn_implementation="eager")
model.eval()   # evaluation mode (we are not training)

print("Model labels:", model.config.id2label)

The model has **two output labels**: `0 = NEGATIVE` and `1 = POSITIVE`.

Let's check the size of this model:

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Number of Transformer layers: {model.config.n_layers}")
print(f"Attention heads per layer:    {model.config.n_heads}")
print(f"Embedding size:               {model.config.dim}")

About **67 million** parameters. (Compare with our Day 2 ANN: ~109 thousand! And GPT-class models have hundreds of **billions**.)

### Step 1️⃣ Tokenization: text → numbers

Computers can't read words, only numbers. The **tokenizer** splits text into **tokens** (words or word-pieces) and maps each to an **ID number** from a fixed vocabulary.

In [ ]:
text = "Transformers are absolutely amazing!"

tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)

ids = tokenizer.convert_tokens_to_ids(tokens)
print("IDs:   ", ids)

print("\nVocabulary size:", tokenizer.vocab_size)

### 🧐 What do you notice?
- Text was **lowercased** (the model is "uncased").
- Rare words get split into **sub-word pieces**. Pieces starting with `##` continue the previous piece. So the model never meets a truly "unknown" word.
- Punctuation like `!` is its own token.

Now see what the model **actually receives**:

In [ ]:
inputs = tokenizer(text, return_tensors="pt")   # "pt" = PyTorch tensors

print("input_ids:     ", inputs["input_ids"])
print("attention_mask:", inputs["attention_mask"])
print()
print("Tokens with special markers:")
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

### 🏷️ Special tokens
- **`[CLS]`** is added at the **start**. It acts as a "summary" token. After the attention layers, its vector holds the meaning of the whole sentence, and **the classifier reads only this token**.
- **`[SEP]`** is added at the **end** to mark the sentence boundary.
- **`attention_mask`** is `1` for real tokens and `0` for padding (used when batching sentences of different lengths).

### Step 2️⃣ Embeddings: IDs → meaning vectors

Each ID is turned into a vector of **768 numbers**. Words with similar meanings end up with similar vectors.

In [ ]:
embedding_layer = model.get_input_embeddings()

with torch.no_grad():
    word_vectors = embedding_layer(inputs["input_ids"])

print("Shape:", word_vectors.shape)
print("   → (batch size, number of tokens, embedding size)")

token_list = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(f"\nFirst 8 numbers of the vector for '{token_list[1]}':")
print(word_vectors[0][1][:8])

### 📐 Do similar words have similar vectors?
Let's measure **cosine similarity** (1 = identical direction, 0 = unrelated) between a few words.

⚠️ *These are the raw "input" embeddings, before attention adds context. They are a rough guide only.*

In [ ]:
def word_vec(word):
    word_id = tokenizer.convert_tokens_to_ids(word)
    with torch.no_grad():
        return embedding_layer(torch.tensor([word_id]))[0]

def similarity(w1, w2):
    return torch.cosine_similarity(word_vec(w1), word_vec(w2), dim=0).item()

pairs = [("good", "great"), ("good", "excellent"), ("good", "bad"),
         ("happy", "sad"), ("king", "queen"), ("king", "banana")]

for a, b in pairs:
    print(f"{a:>8} ↔ {b:<10} similarity = {similarity(a, b):.3f}")

### 🧪 Try it yourself
Add your own pairs, using **single, common lowercase words**.

### Step 3️⃣ Passing through the Transformer layers & Step 4️⃣ Classification

We now run the full model. It returns **logits**, the raw scores for each label. **Softmax** converts these into probabilities that add up to 1 (same softmax we used in the Day 2 digit classifier!).

In [ ]:
text = "I really enjoyed this movie, it was fantastic!"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():                  # no_grad = we're not training, saves memory
    outputs = model(**inputs)

logits = outputs.logits
probs = torch.softmax(logits, dim=-1)

print("Logits (raw scores):", logits)
print("Probabilities:      ", probs)
print()
for i, p in enumerate(probs[0]):
    print(f"  {model.config.id2label[i]:9} → {p.item()*100:.2f}%")

pred = torch.argmax(probs).item()
print("\n🎯 Final prediction:", model.config.id2label[pred])

### 🔁 Recap of what just happened
1. Tokenizer: text → token IDs
2. Embeddings: IDs → 768-number vectors
3. 6 Transformer layers: tokens exchange information through **self-attention**
4. The `[CLS]` vector → classifier → 2 logits → softmax → probabilities

That's exactly what `pipeline` did in one line earlier!

We can also peek at what the vectors look like **as they travel through the layers**:

In [ ]:
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

hidden_states = outputs.hidden_states
print("Number of stages:", len(hidden_states), "(1 embedding output + 6 layers)")
for i, h in enumerate(hidden_states):
    name = "Embeddings" if i == 0 else f"Layer {i}"
    print(f"{name:11} output shape: {tuple(h.shape)}")

The shape stays the same `(1, tokens, 768)` at every stage, but the **numbers inside change**, because each layer rewrites every token's vector using context from the other tokens.

---
## 👀 Part 5: Seeing attention

**Self-attention** is the heart of the Transformer. For each token, the model computes how much it should "pay attention" to every other token.

- **Row** = the token that is *looking*
- **Column** = the token being *looked at*
- **Brighter cell** = more attention

Each layer has **12 attention heads**. Think of them as 12 different "readers", each focusing on a different kind of relationship (grammar, nearby words, sentiment words...).

In [ ]:
sentence = "The movie was not good at all"
inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

attentions = outputs.attentions
token_labels = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Number of layers:", len(attentions))
print("Shape of one layer's attention:", tuple(attentions[0].shape))
print("   → (batch, heads, tokens, tokens)")
print("Tokens:", token_labels)

### 🔥 Attention heatmap
Let's average the 12 heads of a chosen layer and draw the heatmap.

In [ ]:
layer = 5   # 🧪 Try 0 to 5  (0 = first layer, 5 = last layer)

attn = attentions[layer][0].mean(dim=0)    # average over all heads → (tokens, tokens)

plt.figure(figsize=(7, 6))
sns.heatmap(attn.numpy(), xticklabels=token_labels, yticklabels=token_labels,
            cmap="viridis", square=True, cbar_kws={"label": "Attention weight"})
plt.xlabel("Token being looked at")
plt.ylabel("Token that is looking")
plt.title(f"Self-attention, layer {layer + 1} (average of all heads)")
plt.xticks(rotation=45)
plt.show()

### 🧐 How to read it
- Each **row sums to 1**. It shows how one token splits its attention across all tokens.
- Look at the row for **`good`**. Does it attend to **`not`**? That link is how the model understands that *"not good"* is negative!
- In early layers attention is often spread out or local; in later layers it becomes more focused.

### 🧩 All 12 heads of the last layer
Different heads learn different patterns:

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for head, ax in enumerate(axes.flatten()):
    sns.heatmap(attentions[-1][0][head].numpy(), ax=ax, cmap="viridis",
                cbar=False, xticklabels=False, yticklabels=False, square=True)
    ax.set_title(f"Head {head + 1}")
plt.suptitle("Last layer: 12 attention heads for 'The movie was not good at all'", y=1.0)
plt.tight_layout()
plt.show()

Each head has a **different pattern**. That's the power of **multi-head attention**: the model looks at the sentence from 12 different angles at the same time.

### 🧪 Try it yourself
Change `sentence` in the previous cells to something with a pronoun, like *"The dog chased the cat because it was hungry"*, and see what **"it"** attends to.

---
## 🔎 Part 6: Which words drive the decision?

Attention shows where the model *looks*, but what we really want to know is: **which words made it say POSITIVE or NEGATIVE?**

A simple trick: **remove one word at a time** and see how the probability of "POSITIVE" changes.
- If removing a word **drops** the positive score → that word was pushing toward **positive** 🟢
- If removing a word **raises** it → that word was pushing toward **negative** 🔴

In [ ]:
def positive_score(text):
    """Returns the probability (0-1) that the text is POSITIVE."""
    enc = tokenizer(text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.softmax(logits, dim=-1)[0][1].item()   # index 1 = POSITIVE

def word_importance(sentence):
    words = sentence.split()
    base = positive_score(sentence)
    scores = []
    for i in range(len(words)):
        reduced = " ".join(words[:i] + words[i + 1:])
        scores.append(base - positive_score(reduced))
    return words, scores, base

sentence = "The acting was brilliant but the story was boring"   # ✏️ change me

words, scores, base = word_importance(sentence)
colors = ["green" if s > 0 else "red" for s in scores]

plt.figure(figsize=(10, 4))
plt.bar(words, scores, color=colors)
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("Effect on POSITIVE score")
plt.title(f"Word influence  (overall positive probability = {base*100:.1f}%)")
plt.xticks(rotation=30)
plt.show()

🟢 **Green bars** = pushes toward POSITIVE, 🔴 **Red bars** = pushes toward NEGATIVE.

Try these and discuss the results:
- `"I love this phone"`
- `"I do not love this phone"`
- `"This is the worst movie I have ever seen"`

In [ ]:
for s in ["I love this phone", "I do not love this phone"]:
    words, scores, base = word_importance(s)
    print(f"\n{s}   (positive = {base*100:.1f}%)")
    for w, sc in zip(words, scores):
        print(f"   {w:8} {sc:+.3f}")

---
## 🛍️ Part 7: Mini project, analyze customer reviews

A real-world use: a company has hundreds of reviews and wants to know whether customers are happy. We'll let the Transformer do it in seconds.

In [ ]:
reviews = [
    "Battery life is amazing and the screen is gorgeous.",
    "Stopped working after two days. Very disappointed.",
    "Fast delivery and great packaging.",
    "The camera quality is poor in low light.",
    "Best purchase I have made this year!",
    "Customer support never replied to my emails.",
    "Good value for money, I would recommend it.",
    "It is too heavy and the buttons feel cheap.",
    "Absolutely love the design, super comfortable to use.",
    "Terrible experience, I want a refund.",
    "The product is fine, does what it says.",
    "Exceeded my expectations in every way!",
]

df = pd.DataFrame({"review": reviews})
outputs_list = classifier(reviews)

df["sentiment"] = [o["label"] for o in outputs_list]
df["confidence"] = [round(o["score"], 3) for o in outputs_list]

pd.set_option("display.max_colwidth", 80)
df

In [ ]:
counts = df["sentiment"].value_counts()

plt.figure(figsize=(5, 4))
plt.bar(counts.index, counts.values, color=["seagreen" if l == "POSITIVE" else "indianred" for l in counts.index])
plt.title("Customer sentiment")
plt.ylabel("Number of reviews")
plt.show()

print(f"Positive: {counts.get('POSITIVE', 0)}  |  Negative: {counts.get('NEGATIVE', 0)}")
print(f"Average confidence: {df['confidence'].mean()*100:.1f}%")

### 🔍 Find the least-confident predictions
Low confidence = the model is unsure. These are the reviews a human should check:

In [ ]:
df.sort_values("confidence").head(3)

### 🧪 Try it yourself
Replace the `reviews` list with your **own data**, for example:
- Reviews of your college canteen 🍛
- Comments on a YouTube video
- Tweets about a movie

---
## ⚠️ Part 8: Limitations, where Transformers fail

No model is perfect. Great engineers know where their models break. Let's test some tricky sentences.

In [ ]:
tricky = [
    "Oh great, another Monday morning meeting.",            # sarcasm
    "Yeah, right. As if this phone is 'amazing'.",           # sarcasm
    "The movie was not bad at all.",                         # double negation
    "I expected it to be terrible, but it was wonderful.",   # contrast
    "The film was long.",                                    # neutral fact
    "Ye padam romba nalla irukku",                           # another language
]

for text, res in zip(tricky, classifier(tricky)):
    print(f"{res['label']:8} ({res['score']*100:5.1f}%)  →  {text}")

### 🧠 Discussion points

| Issue | Why it happens |
|---|---|
| **Sarcasm** | The words look positive ("great"), but the meaning is negative. Models read the words, not the tone. |
| **Neutral text** | This model only knows two classes. It is forced to pick positive or negative even when text is neutral. |
| **Other languages** | It was trained on **English** movie reviews only. |
| **Bias** | Models learn from human-written data, so they can inherit human biases. |

Let's check for **bias**. We change only a name or group word in an otherwise identical sentence:

In [ ]:
templates = [
    "{} is a doctor.",
    "{} is a good cook.",
    "{} lives in this neighborhood.",
]
subjects = ["He", "She", "A rich man", "A poor man", "An old person", "A young person"]

rows = []
for t in templates:
    for s in subjects:
        sentence = t.format(s)
        rows.append({"sentence": sentence, "positive_score": round(positive_score(sentence), 3)})

bias_df = pd.DataFrame(rows)
bias_df

Do the scores change when only the subject changes? **They shouldn't**, since these sentences are equally neutral. Any difference reveals bias picked up from the training data. This is why **responsible AI** matters: always test models before deploying them to real people.

---
## 🧪 Part 9: Challenge time

Use any of the functions above to try these:

1. **Fool the model:** write a sentence that a human finds clearly positive, but the model calls NEGATIVE (or the reverse). Share it with the class!
2. **Intensity test:** compare `"good"`, `"very good"`, `"extremely good"`, `"the best thing ever"`. Does confidence increase?
3. **Order test:** compare `"Good food but bad service"` with `"Bad service but good food"`. Does word order change the result?
4. **Length test:** paste a long paragraph (more than 500 words). What happens? *(Hint: models have a maximum input length, here 512 tokens.)*

In [ ]:
# 🧪 Scratch cell for your experiments
test_sentences = [
    "Good food but bad service",
    "Bad service but good food",
]

for text, res in zip(test_sentences, classifier(test_sentences)):
    print(f"{res['label']:8} ({res['score']*100:5.1f}%)  →  {text}")

---
## 🏁 Summary

1. **Transformers** read all words at once using **self-attention**, instead of word by word.
2. The pipeline is: **Tokenizer → Embeddings → Transformer layers (attention) → Classifier**.
3. **Pre-trained models** (transfer learning) let us get state-of-the-art results with a few lines of code.
4. **`[CLS]`** summarizes the whole sentence, and its vector is used for classification.
5. We **visualized attention** and measured **word importance** to explain decisions.
6. Models have **limits**: sarcasm, neutral text, other languages and bias.

### 📚 Glossary

| Term | Simple meaning |
|---|---|
| **Token** | A piece of text (word or sub-word) |
| **Tokenizer** | Converts text ↔ token IDs |
| **Embedding** | A list of numbers representing a token's meaning |
| **Self-attention** | Each token decides which other tokens matter to it |
| **Attention head** | One of several parallel "readers" inside a layer |
| **Logits** | Raw output scores before softmax |
| **Softmax** | Turns scores into probabilities that sum to 1 |
| **Pre-trained model** | A model already trained on massive data |
| **Transfer learning** | Re-using a pre-trained model for a new task |
| **Fine-tuning** | Training a pre-trained model a little more on your own data |

### ❓ Think about it
1. Why can a Transformer understand *"not good"* better than a model that just counts positive and negative words?
2. What is the job of the `[CLS]` token?
3. Why is the model confident but wrong on sarcastic sentences?
4. How is **DistilBERT** (understands text) different from **GPT** (generates text)?

### 🚀 Take-home challenges
- Try another model for the same task: `"cardiffnlp/twitter-roberta-base-sentiment-latest"` (adds a **neutral** class).
- Try other `pipeline` tasks: `"summarization"`, `"translation_en_to_fr"`, `"text-generation"`, `"zero-shot-classification"`.
- Collect 20 reviews from a real website and build your own sentiment dashboard.

---
*You've now seen classical ML (Day 1), neural networks (Day 2) and Transformers (Day 3), the full journey toward modern AI like ChatGPT.* 🎓